In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# שלבי ה-Transpiler

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח עם הדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
דף זה מתאר את השלבים של צינור הטרנספילציה המובנה ב-Qiskit SDK. ישנם שישה שלבים:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

הפונקציה [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) יוצרת [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) מובנה המורכב מהשלבים הללו. ה-passes הספציפיים המרכיבים כל שלב תלויים בארגומנטים שמועברים ל-`generate_preset_pass_manager`. ה-`optimization_level` הוא ארגומנט מיקומי שחובה לציין; הוא מספר שלם שיכול להיות 0, 1, 2 או 3. ערכים גבוהים יותר מעידים על אופטימיזציה כבדה יותר אך גם יקרה יותר (ראה [ברירות מחדל וגדרות תצורה לטרנספילציה](defaults-and-configuration-options)).

הדרך המומלצת לטרנספל מעגל היא ליצור staged pass manager מובנה ואז להריץ אותו על המעגל, כמתואר ב-[Transpile with pass managers](transpile-with-pass-managers). עם זאת, חלופה פשוטה יותר אך פחות ניתנת להתאמה אישית היא להשתמש בפונקציה [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). פונקציה זו מקבלת את המעגל ישירות כארגומנט. כמו עם `generate_preset_pass_manager`, ה-passes הספציפיים המשמשים תלויים בארגומנטים, כגון `optimization_level`, שמועברים ל-`transpile`. למעשה, פנימית פונקציית `transpile` קוראת ל-`generate_preset_pass_manager` כדי ליצור staged pass manager מובנה ומריצה אותו על המעגל.
## שלב ה-Init
שלב ראשון זה עושה מעט מאוד כברירת מחדל והוא שימושי בעיקר אם רוצים לכלול אופטימיזציות ראשוניות משלך. מכיוון שרוב אלגוריתמי הפריסה והניתוב מתוכננים לעבוד רק עם שערים של Qubit יחיד ושני Qubits, שלב זה משמש גם לתרגום שערים שפועלים על יותר משני Qubits לשערים שפועלים רק על אחד או שניים.

למידע נוסף על יישום אופטימיזציות ראשוניות משלך לשלב זה, ראה את הסעיף על plugins והתאמה אישית של pass managers.
## שלב ה-Layout
השלב הבא עוסק בפריסה או בקישוריות של ה-Backend שאליו ישלח מעגל. באופן כללי, מעגלים קוונטיים הם ישויות מופשטות שה-Qubits שלהן הם ייצוגים "וירטואליים" או "לוגיים" של Qubits פיזיים בפועל המשמשים בחישובים. כדי לבצע רצף של שערים, יש צורך במיפוי אחד-לאחד מ-Qubits "וירטואליים" ל-Qubits "פיזיים" במכשיר קוונטי ממשי. מיפוי זה מאוחסן כאובייקט `Layout` והוא חלק מהמגבלות המוגדרות בתוך [ארכיטקטורת מערכת ההוראות (ISA)](./transpile#instruction-set-architecture) של Backend.

![תמונה זו ממחישה Qubits הממופים מייצוג החוט לדיאגרמה המייצגת כיצד ה-Qubits מחוברים ב-QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "מיפוי Qubit")

בחירת המיפוי חשובה ביותר למזעור מספר פעולות ה-SWAP הנדרשות למיפוי מעגל הקלט על טופולוגיית המכשיר ולהבטחת השימוש ב-Qubits המכוילים ביותר. בשל חשיבות שלב זה, ה-preset pass managers מנסים מספר שיטות שונות למציאת הפריסה הטובה ביותר. בדרך כלל זה כולל שני שלבים: ראשית, ניסיון למצוא פריסה "מושלמת" (פריסה שאינה דורשת פעולות SWAP), ולאחר מכן, pass היוריסטי שמנסה למצוא את הפריסה הטובה ביותר לשימוש אם לא ניתן למצוא פריסה מושלמת. ישנם שני `Passes` המשמשים בדרך כלל לשלב ראשון זה:

- `TrivialLayout`: ממפה בצורה פשוטה כל Qubit וירטואלי ל-Qubit פיזי עם אותו מספר במכשיר (כלומר, [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). זוהי התנהגות היסטורית המשמשת רק ב-`optimzation_level=1` לניסיון למצוא פריסה מושלמת. אם הדבר נכשל, `VF2Layout` ינסה הבא.
- `VF2Layout`: זהו `AnalysisPass` שבוחר פריסה אידיאלית על ידי טיפול בשלב זה כבעיית איזומורפיזם תת-גרף, הנפתרת על ידי אלגוריתם VF2++. אם נמצאת יותר מפריסה אחת, מופעל היוריסטי ניקוד לבחירת המיפוי עם שגיאה ממוצעת נמוכה יותר.

לאחר מכן לשלב ההיוריסטי, משתמשים בשני passes כברירת מחדל:

- `DenseLayout`: מוצא את תת-הגרף של המכשיר עם הקישוריות הגבוהה ביותר ושיש לו אותו מספר Qubits כמו המעגל (משמש לרמת אופטימיזציה 1 אם יש פעולות בקרת זרימה כגון IfElseOp הקיימות במעגל).
- `SabreLayout`: pass זה בוחר פריסה על ידי התחלה מפריסה אקראית ראשונית והפעלה חוזרת של אלגוריתם `SabreSwap`. pass זה משמש רק ברמות אופטימיזציה 1, 2 ו-3 אם לא נמצאת פריסה מושלמת דרך ה-`VF2Layout` pass. לפרטים נוספים על אלגוריתם זה, עיין בעיתון [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## שלב הניתוב
כדי לממש שער שני-Qubits בין Qubits שאינם מחוברים ישירות במכשיר קוונטי, יש להכניס שער SWAP אחד או יותר למעגל כדי להזיז את מצבי ה-Qubit עד שהם סמוכים במפת השערים של המכשיר. כל שער SWAP מייצג פעולה יקרה ורועשת לביצוע. לפיכך, מציאת מספר מינימלי של שערי SWAP הנדרשים למיפוי מעגל על מכשיר נתון היא שלב חשוב בתהליך הטרנספילציה. ליעילות, שלב זה מחושב בדרך כלל יחד עם שלב ה-Layout כברירת מחדל, אך הם נבדלים לוגית זה מזה. שלב ה-*Layout* בוחר את Qubits החומרה שישמשו, בעוד שלב ה-*Routing* מכניס את מספר שערי ה-SWAP המתאים כדי לבצע את המעגלים באמצעות הפריסה שנבחרה.

עם זאת, מציאת מיפוי ה-SWAP האופטימלי קשה. למעשה, זוהי בעיה NP-hard, ולכן יקרה מדי לחישוב עבור כל המכשירים הקוונטיים הקטנים ביותר ומעגלי הקלט. כדי לעקוף זאת, Qiskit משתמש באלגוריתם היוריסטי סטוכסטי הנקרא `SabreSwap` לחישוב מיפוי SWAP טוב, אך לא בהכרח אופטימלי. השימוש בשיטה סטוכסטית פירושו שהמעגלים שנוצרים אינם מובטחים להיות זהים בריצות חוזרות. אכן, הרצת אותו מעגל שוב ושוב מייצרת התפלגות של עומקי מעגל וספירות שערים בפלט. זו הסיבה שמשתמשים רבים בוחרים להריץ את פונקציית הניתוב (או את ה-`StagedPassManager` כולו) פעמים רבות ולבחור את המעגלים בעומק הנמוך ביותר מהתפלגות הפלטים.

למשל, נקח מעגל GHZ בן 15 Qubits המבוצע 100 פעמים, תוך שימוש ב-`initial_layout` "רע" (מנותק).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeAuckland()

ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/358cfb50-02fc-48f2-a4ec-657837e0c304-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/bb3b8c1f-69fd-4e0c-9b78-9ee67f3361bb-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](./transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

כפי שניתן לראות, מעגל זה צריך לבצע שער שני-Qubits בין Qubits 0 ו-14, שהם רחוקים מאוד זה מזה על גרף הקישוריות. הפעלת מעגל זה מצריכה לכן הכנסת שערי SWAP לביצוע כל שערי שני-ה-Qubits באמצעות ה-`SabreSwap` pass.

שים לב גם שאלגוריתם `SabreSwap` שונה משיטת `SabreLayout` הגדולה יותר בשלב הקודם. כברירת מחדל, `SabreLayout` מריץ גם פריסה וגם ניתוב, ומחזיר את המעגל המומר. הדבר נעשה מכמה סיבות טכניות מיוחדות המפורטות ב-[דף הפניית ה-API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) של ה-pass.
## שלב התרגום
בעת כתיבת מעגל קוונטי, אפשר להשתמש בכל שער קוונטי (פעולה יוניטרית) שרוצים, יחד עם אוסף פעולות שאינן שערים כגון מדידות Qubit או הוראות איפוס. עם זאת, רוב המכשירים הקוונטיים תומכים באופן ילידי רק בקומץ פעולות שערים קוונטיים ולא-שערים. שערים ילידיים אלה הם חלק מהגדרת [ISA](./transpile#instruction-set-architecture) של יעד, ושלב זה של ה-`PassManagers` המובנים מתרגם (או *פורס*) את השערים שצוינו במעגל לשערי הבסיס הילידיים של Backend מוגדר. זהו שלב חשוב, שכן הוא מאפשר לבצע את המעגל על ידי ה-Backend, אך בדרך כלל מוביל לעלייה בעומק ובמספר השערים.

שני מקרים מיוחדים חשובים במיוחד להדגיש, ועוזרים להמחיש מה שלב זה עושה.

1. אם שער SWAP אינו שער ילידי ל-Backend היעד, הדבר דורש שלושה שערי CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2507de9c-1b94-4d56-b5a6-0b2bb1719a80-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

כמכפלה של שלושה שערי CNOT, SWAP הוא פעולה יקרה לביצוע על מכשירים קוונטיים רועשים. עם זאת, פעולות כאלה הכרחיות בדרך כלל לשילוב מעגל בתוך קישוריות שערים מוגבלת של מכשירים רבים. לפיכך, מזעור מספר שערי ה-SWAP במעגל הוא יעד ראשוני בתהליך הטרנספילציה.

2. שער Toffoli, או שער controlled-controlled-not (`ccx`), הוא שער של שלושה Qubits. בהתחשב שמערכת השערים הבסיסית שלנו כוללת רק שערים של Qubit יחיד ושני Qubits, פעולה זו חייבת להיות מפורקת. עם זאת, היא יקרה למדי:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = FakeWashingtonV2()

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4339deb5-4947-493b-8940-e68c91311631-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

עבור כל שער Toffoli במעגל קוונטי, החומרה עשויה לבצע עד שישה שערי CNOT וקומץ שערים חד-Qubit. דוגמה זו מדגימה שכל אלגוריתם המשתמש במספר שערי Toffoli יסתיים כמעגל בעומק גדול ולכן יושפע בצורה ניכרת מרעש.
## שלב האופטימיזציה
שלב זה מתרכז בפירוק מעגלים קוונטיים לסט שערי הבסיס של המכשיר היעד, ועליו להתמודד עם העומק המוגבר משלבי הפריסה והניתוב. למרבה המזל, קיימות רבות שגרות לאופטימיזציה של מעגלים על ידי שילוב או ביטול שערים. במקרים מסוימים, שיטות אלו כה אפקטיביות עד שמעגלי הפלט בעומק נמוך מהקלט, גם לאחר פריסה וניתוב לטופולוגיית החומרה. במקרים אחרים, לא ניתן לעשות הרבה, והחישוב עשוי להיות קשה לביצוע על מכשירים רועשים. שלב זה הוא המקום שבו רמות האופטימיזציה השונות מתחילות להשתנות.

- עבור `optimization_level=1`, שלב זה מכין [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) ו-[`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), המשלבים שרשרות של שערים חד-Qubit ומבטלים שערי CNOT גב-אל-גב.
- עבור `optimization_level=2`, שלב זה משתמש ב-pass [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) במקום `CXCancellation`, שמסיר שערים מיותרים על ידי ניצול יחסי קומוטציה.
- עבור `optimization_level=3`, שלב זה מכין את ה-passes הבאים:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

בנוסף, שלב זה גם מבצע מספר בדיקות סופיות כדי לוודא שכל ההוראות במעגל מורכבות מ-Gates הבסיסיים הזמינים ב-Backend היעד.

הדוגמה הבאה המשתמשת במצב GHZ מדגימה את ההשפעות של הגדרות רמת אופטימיזציה שונות על עומק המעגל וספירת השערים.

> **Note:** פלט הטרנספילציה משתנה בשל מיפוי SWAP הסטוכסטי. לפיכך, המספרים שלהלן ככל הנראה ישתנו בכל פעם שמריצים את הקוד.

![מצב GHZ בן 15 Qubits](../docs/images/guides/transpiler-stages/transpiler-11.avif "מצב GHZ בן 15 Qubits לפני טרנספילציה")

הקוד הבא בונה מצב GHZ בן 15 Qubits ומשווה את `optimization_levels` של הטרנספילציה מבחינת עומק מעגל תוצאתי, ספירות שערים וספירות שערים רב-Qubit.